# 00 — Setup no Colab
Runtime: CPU padrão é suficiente. Ajuste `PROJECT_DIR` se a pasta estiver em outro lugar do Drive.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
import fraud
print(fraud.__file__)

In [ ]:
!pip -q install pytest

In [ ]:
from fraud.config import load_config
from fraud.data import csv_to_parquet, load_parquet

cfg = load_config()
pq = csv_to_parquet(cfg['paths']['raw_csv'], cfg['paths']['parquet'])
df = load_parquet(pq)
print(df.shape)
print(df['isFraud'].value_counts(normalize=True))
df.head()

In [ ]:
from fraud.split import temporal_split
from fraud.features import build_features

s = cfg['split']
train, val, test = temporal_split(df, s['train_end'], s['val_end'])
for name, part in [('treino', train), ('validação', val), ('teste', test)]:
    print(f"{name}: {len(part):,} linhas | fraudes: {int(part['isFraud'].sum()):,} | steps {part['step'].min()}–{part['step'].max()}")

for v in cfg['features']['variants']:
    print(v, build_features(train.head(1000), v).shape)

In [ ]:
!python -m pytest -q